# Sudoku Knowledge Representation & Inference

You will implement three functions -- `build_general_kb`, `build_definite_kb`, `pl_bc_entails` -- and the full-grid solving logic in `sudoku_solver.py`.

This notebook imports and tests those functions. The Streamlit app (`sudoku_app.py`) must import the same implementation from `sudoku_solver.py`; do not copy or rewrite the solver functions inside the app.

**Rules:**

- In `sudoku_solver.py`, import only from `utils.py` and `logic_.py`; do not modify either file.
- Do not duplicate the core solver functions in this notebook or `sudoku_app.py`.
- All other content in this notebook may be edited freely.


## Group 7

**Group Number:** 7

| Member | Student ID |
|---|---|
| ZHANG SHIHAN | A0330526M |
| FANG XUYI | A0353826W |
| CHEN JUNXIAO | A0356288M |
| CHOOG SHENG HONG | Awaiting member information — not yet contacted |

### Planned responsibilities

The following describes the agreed task allocation, not a record of completed contributions. Final contribution statements will be revised after each member's actual work has been confirmed.

- **ZHANG SHIHAN:** Responsible for project coordination and code integration; the Streamlit interface and reasoning trace presentation; application deployment; overall testing of all five Sudoku puzzles; FC/BC runtime comparison; Notebook organization and final submission checks; and conceptual questions Q2 and Q5.
- **CHEN JUNXIAO:** Responsible for encoding the general propositional and Horn knowledge bases, full-grid Sudoku solving with forward chaining, and conceptual questions Q1 and Q4.
- **FANG XUYI:** Responsible for the backward-chaining algorithm and full-grid Sudoku solving, algorithm correctness and cyclic-dependency testing, and conceptual question Q3.
- **CHOOG SHENG HONG:** Not yet contacted. Tasks and actual contributions remain unconfirmed; no completed work is attributed to this member.

**Actual contributions:** Awaiting confirmation based on work actually completed by each member.

**Submission status:** CHOOG SHENG HONG's student ID and the final actual contribution statements are still required. The verified public app URL and deployment checks are recorded below.

## Reproducibility

Python 3.12 was used for the recorded run. `requirements.txt` pins the application
packages; `requirements-notebook.txt` adds the execution dependencies. The supplied
`utils.py` and `logic_.py` are unchanged copies of the uploaded support files.
All core implementation is in `sudoku_solver.py`. Neither this notebook nor the app
contains a second implementation of the core inference functions.


**Execution method:** The notebook is executed from its first cell in a fresh,
independent Jupyter kernel using the project Python environment. All saved
outputs are generated by that execution. The runtime version, assertions,
resource-limited experiments, and measured timings are printed below.


In [1]:
from utils import *
from logic_ import *
import json
import time
import importlib
import sudoku_solver

# Reload so edits made to sudoku_solver.py are picked up when this cell is rerun.
importlib.reload(sudoku_solver)

from sudoku_solver import (
    atom,
    build_general_kb,
    build_definite_kb,
    solve_full_grid_fc,
    pl_bc_entails,
    solve_full_grid_bc,
)


## Loading a puzzle from JSON

Puzzles are provided as JSON, not embedded in this notebook. Each file looks like:

```json
{
  "n": 9, "box_h": 3, "box_w": 3,
  "puzzles": [
    {
      "givens": {"1_1": 3, "2_3": 1, ...},
      "given_count": 28,
      "solution": {"1_1": 3, "1_2": 4, ...}
    },
    ...
  ]
}
```

`"r_c"` string keys map to the value at row `r`, column `c` (1-indexed). `given_count` is exactly how many cells are given. `solution` is included so you can check your own work as you go, but your functions must not read `solution` to answer a query, they should only read `givens`.

In [2]:
#do not change this function, it is used to load the puzzle pool from a json file
def load_pool(path):
    with open(path) as f:
        raw = json.load(f)
    puzzles = []
    for p in raw['puzzles']:
        givens = {tuple(int(x) for x in k.split('_')): v for k, v in p['givens'].items()}
        solution = {tuple(int(x) for x in k.split('_')): v for k, v in p['solution'].items()}
        puzzles.append({'givens': givens, 'solution': solution, 'given_count': p['given_count']})
    return raw['n'], raw['box_h'], raw['box_w'], puzzles

n, box_h, box_w, puzzle_pool = load_pool('puzzles.json')
print(f'{len(puzzle_pool)} puzzles loaded, {n}x{n} grid, {box_h}x{box_w} boxes')
print('given_count values:', sorted(p['given_count'] for p in puzzle_pool))

# Pick one puzzle to work with through the rest of this notebook.
puzzle = puzzle_pool[0]
givens = puzzle['givens']
print(f"working puzzle has {puzzle['given_count']} givens")
for r in range(1, n + 1):
    print([givens.get((r, c), '.') for c in range(1, n + 1)])

5 puzzles loaded, 9x9 grid, 3x3 boxes
given_count values: [30, 33, 36, 39, 42]
working puzzle has 30 givens
['.', 3, '.', '.', '.', '.', '.', '.', '.']
[2, '.', '.', '.', '.', 7, 8, 6, '.']
[5, 8, '.', 2, 6, '.', '.', 3, '.']
[7, 5, '.', '.', '.', '.', '.', 8, '.']
['.', '.', '.', '.', 7, '.', 5, '.', 4]
['.', '.', '.', 5, 3, '.', '.', 9, 6]
['.', 1, 2, '.', '.', 9, '.', '.', '.']
[6, 4, '.', '.', 5, 8, 9, '.', '.']
['.', '.', '.', '.', 2, 3, '.', '.', '.']


Notice: `pl_fc_entails`/`pl_resolution`/`tt_entails` are all given to you, already implemented, in `logic_.py`. The one algorithm you write yourself in this assignment is **backward chaining** (`pl_bc_entails`) -- see Task 2.

## Define Symbols

Two families of propositional symbols, for row $r$, column $c$, value $v$ (all ranging over $1$ to $n$):

| Symbol | Meaning |
|---|---|
| $\mathit{Is}_{rcv}$ | cell $(r,c)$ has value $v$ |
| $\mathit{Not}_{rcv}$ | cell $(r,c)$ does **not** have value $v$ |

For each representation in Task 1, determine which of these two families is actually needed.

Hint: consider what form a definite (Horn) clause must take, and what `PropDefiniteKB.tell()` will accept.

### Helper function

`atom(prefix, r, c, v)`, where prefix can be either `Is` or `Not`, is a naming helper so propositional symbols need not be typed. It is provided for you in `sudoku_solver.py`; do not change it.

In code, $\mathit{Is}_{rcv}$ and $\mathit{Not}_{rcv}$ are written as single-word symbol names, e.g. `Is3_2_4` and `Not3_2_4`: the prefix is followed immediately by `r`, then by `c` and `v` separated by underscores. No separator is needed between the prefix and `r` since `expr()` only requires a symbol name to start with an uppercase letter. So `Is3_2_4` is parsed as one valid symbol.


In [3]:
# atom() is provided in sudoku_solver.py and imported above.


## Part A: Design the Sudoku Solver

### A.1) Knowledge Representation - Build KB

Every well-posed Sudoku puzzle satisfies exactly these conditions:

- Each cell is assigned **at least one** value from $\{1, \dots, n\}$.
- Each cell is assigned **at most one** value from $\{1, \dots, n\}$, i.e., it cannot hold two different values at once.
- No two cells in the same row hold the same value.
- No two cells in the same column hold the same value.
- No two cells in the same box hold the same value.
- The **givens** cells hold their stated values.

Formalize these Sudoku constraints as propositional logic, in **two** representations:

**(a) General clauses -  `build_general_kb`.**

Encode each of the following directly: no restriction here; you may use arbitrary disjunctions of positive or negated literals. Must return a `PropKB`. 

**(b) Definite (Horn) clauses:- `build_definite_kb`.** Must return a `PropDefiniteKB`. Recall a definite clause is a disjunction of literals with exactly one *positive* literal.

Equivalently written as an implication whose conclusion is a single positive literal and whose premises are a conjunction of positive literals: `P1 & P2 & ... & Pk ==> Q`.

`PropDefiniteKB.tell()` will reject anything else.

Both functions take the puzzle's givens as fixed facts.

- **Implement `build_general_kb()` and `build_definite_kb()` in sudoku_solver.py**
- **Rerun the import cell near the top of this notebook after making changes.**
- **Explain your representation in Conceptual Question 1 below.**

In [4]:
general_kb = build_general_kb(n, box_h, box_w, givens)
definite_kb = build_definite_kb(n, box_h, box_w, givens)
assert isinstance(general_kb, PropKB)
assert isinstance(definite_kb, PropDefiniteKB)
assert all(is_definite_clause(c) for c in definite_kb.clauses)
print('General CNF clauses:', len(general_kb.clauses))
print('General proposition count:', len(prop_symbols(associate('&', general_kb.clauses))))
print('Definite clauses including facts:', len(definite_kb.clauses))
print('Given facts:', len(givens))

General CNF clauses: 10317
General proposition count: 729
Definite clauses including facts: 21171
Given facts: 30


### A.2) Solve the Puzzle

**(a) Resolution and model checking on the general representation.** Using `build_general_kb` and the library's `pl_resolution` / `tt_entails`, try to solve the puzzle, i.e., for each cell, determine which value is entailed. Attempt this in the code cell below: it is provided commented out, because both are sound and complete on `build_general_kb`'s output but neither scales to the full grid, and it is expected to hang or take an impractically long time. Uncomment a few lines at a time and give each at most about 30 seconds; use Kernel > Interrupt if it hasn't returned by then, and note what you observed.

Explain in your own words: what specifically makes `pl_resolution`'s cost grow out of control here, and separately, what makes `tt_entails`'s cost grow out of control? A simple complexity argument for each is the expected answer.

**Use your observations in Conceptual Question 2 below.**


In [5]:
import subprocess
import sys
import time
import psutil
from IPython.display import display, Markdown

# Run the supplied algorithms unchanged in separate processes so a slow query
# can be stopped without interrupting the rest of the Notebook.
worker = """
import json, sys
from sudoku_solver import build_general_kb, atom
from logic_ import pl_resolution, tt_entails, associate
pool = json.load(open('puzzles.json'))
givens = {tuple(map(int, k.split('_'))): v
          for k, v in pool['puzzles'][0]['givens'].items()}
kb = build_general_kb(pool['n'], pool['box_h'], pool['box_w'], givens)
(r, c), v = next(iter(sorted(givens.items())))
query = atom('Is', r, c, v)
print('Algorithm started', flush=True)
if sys.argv[1] == 'resolution':
    print(pl_resolution(kb, query), flush=True)
else:
    print(tt_entails(associate('&', kb.clauses), query), flush=True)
"""

experiment_results = []
for method in ('resolution', 'model checking'):
    start = time.perf_counter()
    with subprocess.Popen([sys.executable, '-c', worker, method],
                          stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True) as proc:
        monitor = psutil.Process(proc.pid)
        status = None
        while proc.poll() is None:
            if time.perf_counter() - start >= 30:
                status = 'stopped at 30-second wall-time limit'
            try:
                # A short, portable guard avoids exhausting this computer's RAM.
                if monitor.memory_info().rss > 768 * 1024**2:
                    status = 'stopped after resident memory exceeded 768 MiB'
            except psutil.NoSuchProcess:
                break
            if status:
                proc.kill()
                break
            time.sleep(0.05)
        stdout, stderr = proc.communicate()
        status = status or ('returned ' + stdout.strip().splitlines()[-1]
                            if proc.returncode == 0 else 'process error: ' + stderr.strip()[-300:])
    result = dict(method=method, status=status,
                  algorithm_started='Algorithm started' in stdout,
                  wall_seconds=round(time.perf_counter() - start, 3))
    experiment_results.append(result)
    print(result, flush=True)
print('A timeout or memory stop is not a True/False entailment result.')


{'method': 'resolution', 'status': 'stopped at 30-second wall-time limit', 'algorithm_started': True, 'wall_seconds': 30.025}


{'method': 'model checking', 'status': 'stopped at 30-second wall-time limit', 'algorithm_started': True, 'wall_seconds': 30.028}


A timeout or memory stop is not a True/False entailment result.


### Observation notes

Record what you observed from the resolution/model-checking experiment here. Use these observations when answering Conceptual Question 2 below.


In [6]:
for result in experiment_results:
    display(Markdown(f"- **{result['method'].title()}:** {result['status']}; "
                     f"observed wall time {result['wall_seconds']:.3f} s."))
count_with_query = len(general_kb.clauses) + 1
pairs = count_with_query * (count_with_query - 1) // 2
display(Markdown(f"The first resolution round requests **{pairs:,} clause pairs**. "
                 "The model checker has **729 propositional variables**, giving up to "
                 r"$2^{729}$ truth assignments. Only one atomic query was attempted "
                 "per method; the observed limit already prevents a practical full-grid solve."))


- **Resolution:** stopped at 30-second wall-time limit; observed wall time 30.025 s.

- **Model Checking:** stopped at 30-second wall-time limit; observed wall time 30.028 s.

The first resolution round requests **53,225,403 clause pairs**. The model checker has **729 propositional variables**, giving up to $2^{729}$ truth assignments. Only one atomic query was attempted per method; the observed limit already prevents a practical full-grid solve.

**(b) Forward chaining on the full grid --** Implement `solve_full_grid_fc()` in sudoku_solver.py. Using your above `build_definite_kb` and the library's `pl_fc_entails` (no need to reimplement them), solve the whole puzzle. Find the value that's entailed for every cell. Reconstruct and display the solved grid.

**(c) Backward chaining -- implement it yourself.**
```python
pl_bc_entails(kb, query) -> bool
```
Implement `pl_bc_entails()` in sudoku_solver.py. Start from the query and recursively try to prove each premise of a rule whose conclusion matches the current goal, bottoming out at known facts. Your function must agree with `pl_fc_entails` on every cell/value pair in this puzzle including correctly returning `False` for values that are *not* part of the solution.

**(d) Backward chaining on the full grid --** Implement `solve_full_grid_bc` in sudoku_solver.py. Using your above `build_definite_kb` and your own `pl_bc_entails`, solve the whole puzzle the same way `solve_full_grid_fc` does: for every cell, try each candidate value until `pl_bc_entails` confirms one. Time both `solve_full_grid_fc` and `solve_full_grid_bc` on the same puzzle and compare. Use your measured result where relevant in Conceptual Question 5.


In [7]:
# The single implementation is imported from sudoku_solver.py.
# Prove one initially empty cell and inspect a real proof (no solution used).
from sudoku_solver import explain_bc
empty_cell = next((r, c) for r in range(1, n + 1) for c in range(1, n + 1)
                  if (r, c) not in givens)
for candidate in range(1, n + 1):
    query = atom('Is', *empty_cell, candidate)
    if pl_bc_entails(definite_kb, query):
        entailed, proof = explain_bc(definite_kb, query)
        print('Proved cell', empty_cell, '=', candidate)
        print('Supporting proof steps:', len(proof))
        break

Proved cell (1, 1) = 1
Supporting proof steps: 200


### Verifying the algorithms

Uncomment the following block of code to validate your code.

In [8]:
from statistics import median
import platform

print('Runtime:', platform.python_version(), platform.platform(), flush=True)
validation_rows = []
for i, entry in enumerate(puzzle_pool, 1):
    start = time.perf_counter()
    solved_fc = solve_full_grid_fc(n, box_h, box_w, entry['givens'])
    fc_seconds = time.perf_counter() - start
    start = time.perf_counter()
    solved_bc = solve_full_grid_bc(n, box_h, box_w, entry['givens'])
    bc_seconds = time.perf_counter() - start
    # The answer key is used ONLY in assertions, after both solvers finish.
    assert solved_fc == solved_bc == entry['solution']
    kb = build_definite_kb(n, box_h, box_w, entry['givens'])
    for (r, c), correct in entry['solution'].items():
        for candidate in range(1, n + 1):
            query = atom('Is', r, c, candidate)
            expected = candidate == correct
            assert pl_bc_entails(kb, query) == expected
            assert pl_fc_entails(kb, query) == expected
    print(f'Puzzle {i}: FC and BC agree with all 729 correct/incorrect candidates', flush=True)
    row = {'puzzle': i, 'givens': len(entry['givens']),
           'fc_seconds': fc_seconds, 'bc_seconds': bc_seconds,
           'bc_queries_checked': n ** 3, 'fc_queries_checked': n ** 3}
    validation_rows.append(row)
    print(f"Puzzle {i}: {len(entry['givens'])} givens; FC {fc_seconds:.3f}s; "
          f"BC {bc_seconds:.3f}s; complete grid and 729 FC/BC candidate pairs PASS", flush=True)

print('All five puzzles: 3,645 FC/BC candidate pairs PASS', flush=True)

# Three measured full-grid runs of the same puzzle, fresh KB per call.
fc_runs = [validation_rows[0]['fc_seconds']]
bc_runs = [validation_rows[0]['bc_seconds']]
for repetition in range(2):
    for solve, results in ((solve_full_grid_fc, fc_runs), (solve_full_grid_bc, bc_runs)):
        start = time.perf_counter()
        answer = solve(n, box_h, box_w, givens)
        results.append(time.perf_counter() - start)
        assert answer == puzzle['solution']
    print(f'Additional timing repeat {repetition + 1} finished', flush=True)
fc_time, bc_time = median(fc_runs), median(bc_runs)
print('FC times (s):', [round(t, 3) for t in fc_runs])
print('BC times (s):', [round(t, 3) for t in bc_runs])
print(f'Median FC={fc_time:.3f}s; median BC={bc_time:.3f}s')
print('Solved first puzzle:')
solved = solve_full_grid_bc(n, box_h, box_w, givens)
for r in range(1, n + 1):
    print([solved[r, c] for c in range(1, n + 1)])

Runtime: 3.12.14 Windows-11-10.0.26200-SP0


Puzzle 1: FC and BC agree with all 729 correct/incorrect candidates


Puzzle 1: 30 givens; FC 10.018s; BC 0.362s; complete grid and 729 FC/BC candidate pairs PASS


Puzzle 2: FC and BC agree with all 729 correct/incorrect candidates


Puzzle 2: 33 givens; FC 10.833s; BC 0.559s; complete grid and 729 FC/BC candidate pairs PASS


...

Additional timing repeat 1 finished


Additional timing repeat 2 finished


FC times (s): [10.018, 16.053, 15.718]
BC times (s): [0.362, 0.438, 0.423]
Median FC=15.718s; median BC=0.423s
Solved first puzzle:


[1, 3, 6, 9, 8, 5, 4, 7, 2]


[2, 9, 4, 3, 1, 7, 8, 6, 5]
[5, 8, 7, 2, 6, 4, 1, 3, 9]
[7, 5, 1, 4, 9, 6, 2, 8, 3]
[3, 6, 9, 8, 7, 2, 5, 1, 4]
[4, 2, 8, 5, 3, 1, 7, 9, 6]
[8, 1, 2, 6, 4, 9, 3, 5, 7]
[6, 4, 3, 7, 5, 8, 9, 2, 1]
[9, 7, 5, 1, 2, 3, 6, 4, 8]


## Part B: Conceptual Questions

All five answers below describe the submitted implementation and the executed experiments.


### 1. Detailed Representation Strategy: General vs. Definite (Horn) Encoding

Explain in detail how you formalized the Sudoku puzzle constraints into propositional logic across both Knowledge Base representations:

**(a) General KB Strategy (`build_general_kb`):** Detail how standard Sudoku rules (e.g., at-least-one value per cell, at-most-one value per cell, row/column/box uniqueness) are directly translated into Conjunctive Normal Form (CNF) clauses without structural restrictions.

**(b) Definite KB Strategy (`build_definite_kb`):** Definite/Horn clauses strictly permit at most one positive literal per clause, prohibiting disjunctive constraints like $(Is_{r,c,1} \lor Is_{r,c,2} \lor Is_{r,c,3} \lor ... \lor Is_{r,c,n})$. Explain step-by-step how your encoding deals with this issue.


**General CNF representation.** Let $I_{rcv}$ mean that cell $(r,c)$ contains $v$.
Each cell receives one clause $I_{rc1}\lor\cdots\lor I_{rcn}$. For every distinct
pair $u<v$ in that cell we add $\neg I_{rcu}\lor\neg I_{rcv}$. Together these
say exactly one value is assigned. For two different cells $a,b$ sharing a row,
column, or box, add $\neg I_{av}\lor\neg I_{bv}$ for each $v$. Finally, each
given becomes a positive unit clause. These clauses directly encode conditions
1–6. Calling `PropKB.tell` converts to CNF; our clauses are already in CNF.
We deduplicate overlapping row/box or column/box constraints by processing each
unordered peer-cell pair once. The 9×9 encoding uses 729 Is atoms, 81 at-least-one
clauses, 2,916 within-cell exclusions, 7,290 peer exclusions, and the given facts.

**Definite representation.** A definite clause has exactly one positive literal
(or equivalently a positive conjunctive body implying one positive head).
General Horn clauses allow at most one, but the supplied `PropDefiniteKB`
requires definite clauses. A multi-positive at-least-one disjunction cannot be
directly inserted. We introduce the positive atom $N_{rcv}$, meaning that $v$
has been explicitly excluded; it is separate from logical negation.

The operational encoding is:

1. Given $(r,c)=v$: assert $I_{rcv}$.
2. Same cell, $u\ne v$: $I_{rcv}\Rightarrow N_{rcu}$.
3. Distinct peer cells $a,b$: $I_{av}\Rightarrow N_{bv}$ and
   $I_{bv}\Rightarrow N_{av}$ for every value $v$.
4. Last candidate: $\bigwedge_{u\ne v}N_{rcu}\Rightarrow I_{rcv}$.

For example, once a cell's eight other numbers are excluded, rule 4 places the
remaining number. Rules 2–3 propagate the at-most-one and row/column/box
constraints; rule 4 is the elimination consequence of the at-least-one constraint.
All rule premises and heads are positive atoms, so `tell` accepts them. For 9×9
there are 5,832 same-cell rules, 14,580 directed peer rules, 729 last-candidate
rules, and the givens.

**Scope of the representation.** These are sound deduction rules relative to the
Sudoku constraints, not a logically equivalent Horn translation of the full CNF.
They cannot express arbitrary choices or guarantee a placement in every cell of
an arbitrary puzzle. Both inference algorithms are complete for this Horn KB,
but this particular rule set is not a complete method for all Sudoku puzzles.
If elimination stalls, the solver reports an unresolved cell rather than guessing.
A failed Is query never causes a Not fact to be asserted.

### 2. Theoretical Completeness vs. Computational Tractability

Model checking and resolution-refutation are sound and complete—they are guaranteed to terminate with a correct answer for any propositional KB. Despite this guarantee, explain whether you would use either as the default algorithm for solving Sudoku puzzles. *(Hint: Consider space/time complexity and state-space growth, and use your observations from the experiment above where relevant.)*


Neither supplied algorithm is a practical default for the 9×9 board. Soundness
and completeness describe logical correctness, not a usable time or memory bound.
The resource-limited observations above measure one atomic query, already enough
to demonstrate the implementation costs.

- **Model checking (`tt_entails`).** There are 729 Boolean Is variables, so exhaustive
  enumeration has up to $2^{729}\approx2.82\times10^{219}$ leaves. In this supplied
  implementation `tt_check_all` evaluates the KB only after assigning every variable;
  it does not prune partial assignments using Sudoku constraints. Each leaf also
  has a nontrivial formula-evaluation cost. Short-circuiting can sometimes find a
  countermodel early, but a genuinely entailed query still requires exhaustive
  checking. A 30-second cutoff is not a logical False result.
- **Resolution (`pl_resolution`).** The code first materializes every unordered
  pair among $m$ clauses, using $m(m-1)/2=O(m^2)$ list entries before resolving
  the first pair. With 10,317 original clauses and the negated query, the first
  round alone has 53,225,403 pairs. This creates a large memory demand even for a
  query whose value is given. Subsequent rounds can add many resolvents and repeat
  the pair construction. The worst-case number of distinct propositional clauses
  also grows exponentially with the number of variables.

Each original algorithm is attempted in a separate process for at most about
30 seconds, including startup and KB construction. A small monitoring loop also
stops a child process if its resident memory exceeds 768 MiB, to protect the
computer. This is a sampled stopping threshold, not an operating-system memory
cap, so usage can briefly exceed it between checks. The observations above report
which limit was actually reached; neither stopping condition is a logical False.
These are bounded attempts, not unrestricted completion times. The supplied
algorithms and logic library are unchanged. Full-grid solving instead uses the
elimination Horn KB.


### 3. Backward Chaining: Design, Pseudocode, and Challenges

Write pseudocode for `pl_bc_entails(kb, query)`, the backward-chaining algorithm you implemented. Show, at a level of detail that reveals the algorithm's structure (not full Python), how the function checks whether the query is already a known fact, finds candidate rules whose conclusion matches the current goal, recursively proves each premise of such a rule, and combines results—both across the premises of one rule and across multiple candidate rules—to reach a single boolean answer.

Then, in your own words, discuss the design challenges you had to work through to make your algorithm both correct and guaranteed to terminate on every puzzle, and explain how your pseudocode addresses them.


**Design.** This implementation uses goal-directed backward chaining with tabling.
For the current goal it looks up rules with that conclusion (OR alternatives).
Every premise of one rule must be proved (AND). Known facts are immediate proofs.
A repeated goal in a cycle is suspended rather than accepted or permanently failed.
When a premise is proved, dependent suspended rules are resumed. This completion
step propagates answers only within the subgraph demanded by backward expansion;
it is not a call to the supplied forward-chaining solver.

```text
INITIALIZE(KB):
    index rules by conclusion
    PROOFS := given facts
    EXPANDED := empty set
    WAITING := empty map from premise to dependent rules
    ANSWERS := empty queue; DEFERRED := empty stack

PROVE(query):
    if query is in PROOFS: return True
    push query on DEFERRED
    while DEFERRED is not empty:
        EXPAND(pop DEFERRED, depth = 0)
        RESUME()
    return query is in PROOFS

EXPAND(goal, depth):
    if goal is proved or already expanded: return
    if depth reaches 64:
        defer goal; return                 # resume later, not a failure
    mark goal expanded
    for each rule premises => goal:        # OR over rules
        missing := distinct premises without a proof
        remaining[rule] := size(missing)
        if missing is empty:
            RECORD(goal, rule); RESUME(); stop trying alternatives
        register rule as waiting on every missing premise
        for premise in missing:           # AND over premises
            EXPAND(premise, depth + 1)
            RESUME()
        if goal now has a proof: stop trying alternatives

RECORD(goal, rule):
    if goal has no proof yet:
        save the rule and its proven premises
        enqueue goal in ANSWERS

RESUME():
    while ANSWERS is not empty:
        p := pop ANSWERS
        for each rule waiting on p:
            decrement remaining[rule]
            if remaining[rule] is zero: RECORD(rule.head, rule)
        remove the processed waiting list for p
```

**Correctness and termination challenges.** A cycle such as $A\Rightarrow B$,
$B\Rightarrow A$ cannot create a proof by itself. Marking an active goal as True
would be unsound. Permanently caching a provisional False can also miss a later
proof through another rule, such as $C\Rightarrow A$ with fact $C$. Our tables
keep these goals pending; an independently established fact can reactivate their
rules. A rule fires only after all distinct premises have actual proofs, and each
atom receives at most one proof. Thus every recorded deduction is sound.

For completeness, backward expansion requests the rule dependencies needed by the
query. Once expansion and answer propagation stop, every demanded rule whose body
is true has fired. If the query is still absent, it has no finite fact-grounded proof
in this finite definite KB. Each goal is expanded at most once and each waiting
premise is consumed at most once, so cycles do not create endless work. Recursive
expansion is chunked at depth 64 and resumes through a deferred stack, preventing
Python's call-stack limit from terminating a logically finite proof search.

Proven subgoals are shared across queries on the same KB. `tell` and `retract`
invalidate the tables through a version counter (ordinary supplied KBs use a clause
snapshot). The cache must never be reused after an untracked direct edit of an
indexed KB's `clauses` list. Proofs record dependencies at the moment the conclusion
is established; the app displays this actual proof DAG in prerequisite-first order.

### 4. Expressive Limits of Horn Logic

Named elimination techniques such as Naked Pairs and X-Wing can, in fact, be encoded as definite clauses, using the same `Is`/`Not` vocabulary as your `build_definite_kb`. Work out how you would encode one of these techniques as definite clauses, and discuss the consequences of doing so.


I use **Naked Pairs**. Consider distinct cells $a,b$ in the same unit (row, column,
or box), and values $p\ne q$. Suppose every other value has been explicitly excluded
from both cells. Define the positive conjunction

$$D_{a,b,p,q}=\bigwedge_{v\notin\{p,q\}}(N_{av}\land N_{bv}).$$

For each other cell $c$ in that unit, add two **separate** definite rules:

$$D_{a,b,p,q}\Rightarrow N_{cp},\qquad D_{a,b,p,q}\Rightarrow N_{cq}.$$

For example, let $a=(1,1)$, $b=(1,2)$, and $\{p,q\}=\{2,7\}$. The body is the
conjunction of the 14 positive Not atoms excluding values $1,3,4,5,6,8,9$ from
each of those two cells. It implies `Not1_3_2` in one rule and `Not1_3_7` in another,
and similarly for columns 4–9 in row 1. Each rule has exactly one positive head.
The body is written out using the existing atoms; $D$ above is only notation,
not a required new proposition.

Why this is sound: in a valid Sudoku the two cells each need a value, both values
come from $\{p,q\}$, and the cells cannot have equal values because they share a
unit. They must therefore use $p$ and $q$ in some order. Neither value can appear
in another cell of that unit. It is enough that their candidate sets are subsets
of this pair; we do not infer unproven candidates by treating missing Not facts as
false, and we do not guess which cell takes which value.

These rules can trigger extra eliminations when last-candidate reasoning alone
stalls. The cost is a much larger KB: a naive 9×9 grounding considers 27 units,
$\binom92$ pairs of cells, $\binom92$ pairs of values, 7 other cells, and 2 heads:
$27\times36\times36\times7\times2=489{,}888$ candidate rules before deduplication.
Each body has 14 premises. This increases construction time, memory, and premise
processing. Rule indexing or generating relevant rules only when needed can help.
Even this extension would not make the rule set sufficient for every Sudoku.
The submitted solver intentionally implements the required elimination and
last-candidate rules; this question develops the extension without silently
changing the measured baseline.

### 5. Data-Driven vs. Goal-Driven Performance

Forward chaining (data-driven) and backward chaining (goal-driven) are both sound and complete for Horn KBs, but their execution runtimes vary depending on the target query.

**(a)** Describe a scenario—in terms of total KB size versus the query-relevant subset—where backward chaining is significantly faster than forward chaining.

**(b)** Describe a scenario where backward chaining offers no performance advantage, or performs worse than forward chaining.

Use your measured forward- vs. backward-chaining result where relevant.


**When BC can be faster.** If a large Horn KB contains many unrelated rule groups
but a query depends on only a small chain of facts, backward search follows that
chain and avoids expanding the unrelated groups. A query that is already a fact
can finish immediately after indexing. Index construction still reads the KB once,
so this advantage is especially useful across repeated queries.

**When BC has no advantage or is slower.** If most rules are relevant to the goal,
its demanded subgraph can be almost the entire KB. Dense cycles require table and
waiting-list management. For many queries with overlapping dependencies, a forward
engine that computes and retains the closure once can be better than restarting
backward search for each goal. If a query is unprovable, a backward method may have
to exhaust all relevant alternative rules before returning False. There is no
universal winner based only on the labels "forward" and "backward".

**Our measured comparison.** The executed table and timing output above use the
same first puzzle for three runs per method and report medians. Each full-grid call
starts with a fresh KB, includes construction time, and omits printing, UI work,
and assertions from its timed interval. Both methods try values in ascending order
and copy given facts directly; neither reads the solution. The other four puzzles
also have measured full-grid results and correctness assertions.

The supplied FC function rebuilds its counters and agenda for each candidate
query. Its premise lookup is indexed by a subclass, but the library algorithm itself
is unchanged. Our BC retains positive subgoal proofs and completed dependency
tables across cell queries on the same KB. Therefore the comparison measures these
**specific implementations**, including this reuse difference, rather than isolating
search direction under identical caching. A retained-closure FC implementation
would be a different experiment. Timings also depend on hardware and runtime load.
These measurements were made on a shared local workstation without CPU isolation. Background activity and the observed run-to-run variation limit benchmark precision.


In [9]:
display(Markdown(
    f"**Measured result for puzzle 1 ({len(givens)} givens):** "
    f"median FC = **{fc_time:.3f} s**, median BC = **{bc_time:.3f} s** "
    f"over three full-grid runs each. FC/BC time ratio = **{fc_time / bc_time:.2f}**. "
    "These numbers describe the implementations and caching policies above."))


**Measured result for puzzle 1 (30 givens):** median FC = **15.718 s**, median BC = **0.423 s** over three full-grid runs each. FC/BC time ratio = **37.19**. These numbers describe the implementations and caching policies above.

### Additional correctness checks

The following checks cover ungrounded cycles, a cycle with an external proof,
cache invalidation after retracting a fact, and an underdetermined puzzle.
They check failure modes that a solved-grid comparison alone would miss.


In [10]:
test_kb = PropDefiniteKB()
for clause in ['A ==> B', 'B ==> A', 'C ==> A']:
    test_kb.tell(expr(clause))
assert not pl_bc_entails(test_kb, expr('B'))
test_kb.tell(expr('C'))
assert pl_bc_entails(test_kb, expr('B'))
test_kb.retract(expr('C'))
assert not pl_bc_entails(test_kb, expr('B'))
try:
    solve_full_grid_bc(4, 2, 2, {})
except ValueError as error:
    print('Underdetermined grid:', error)
else:
    raise AssertionError('The solver must not guess an unproved placement.')
small_kb = build_general_kb(2, 1, 2, {(1, 1): 1})
assert tt_entails(associate('&', small_kb.clauses), atom('Is', 2, 2, 1))
assert not tt_entails(associate('&', small_kb.clauses), atom('Is', 2, 2, 2))
assert pl_resolution(small_kb, atom('Is', 2, 2, 1))
print('Cycle, cache, unresolved-grid, and small-grid inference checks PASS.')

# Q needs B and A; B initially depends on the active A, which has another proof.
cycle_kb = PropDefiniteKB()
for clause in ['(A & B) ==> Q', 'B ==> A', 'A ==> B', 'C ==> A', 'C']:
    cycle_kb.tell(expr(clause))
assert pl_bc_entails(cycle_kb, expr('Q'))
assert pl_bc_entails(cycle_kb, expr('B'))
assert pl_fc_entails(cycle_kb, expr('Q'))
empty_kb = build_definite_kb(4, 2, 2, {})
assert not pl_bc_entails(empty_kb, atom('Is', 1, 1, 1))
assert not pl_bc_entails(empty_kb, atom('Not', 1, 1, 1))
print('No premature negative caching; lack of proof is not explicit exclusion: PASS.')


Underdetermined grid: Cell (1, 1) is unresolved by elimination and last-candidate rules; no value was guessed.
Cycle, cache, unresolved-grid, and small-grid inference checks PASS.
No premature negative caching; lack of proof is not explicit exclusion: PASS.


## Part C: Streamlit Integration

Wrap your Sudoku solver and inference logic into an interactive Streamlit application, `sudoku_app.py`. Your application must implement the following:

**1. Puzzle selection & visual board display**
- An interactive selector/dropdown to pick any puzzle from `puzzles.json`.
- A visual rendering of the grid that clearly distinguishes the initial *givens* from empty cells.

**2. Full-grid auto-solver, with algorithm selection**
- A control (e.g. radio buttons) letting the user choose forward chaining (`solve_full_grid_fc`) or backward chaining (`solve_full_grid_bc`) before solving.
- A button that solves the full grid with the chosen algorithm, renders the solved state, and displays how long the solve took -- so the timing gap from task (d) is visible in the app, not just in the notebook.

**3. Targeted cell entailment query**
- Inputs for row ($r$), column ($c$), and value ($v$).
- A button that checks whether $\mathit{Is}_{rcv}$ is entailed (using `pl_bc_entails`) and displays the boolean verdict (`True` / `False`).

**4. Reasoning trace ("tutor mode")**
- Instrument your chosen inference algorithm (forward or backward chaining) to record the reasoning steps it takes while answering a query.
- Present that trace in a human-readable format -- not a raw Python string or internal symbol dictionary. For example: expandable cards/accordions showing the rule-firing sequence (*"Inferred $\mathit{Not}_{1,2,3}$ because row 1 already contains value 3"* $\implies$ *"Deduce $\mathit{Is}_{1,2,4}$ as the last remaining candidate"*), or plain-English sentences explaining each elimination by row, column, or box constraint.

Import `atom`, `build_definite_kb`, `build_general_kb`, `solve_full_grid_fc`, `solve_full_grid_bc`, and `pl_bc_entails` from `sudoku_solver.py`. Do not copy or rewrite these core functions in the Streamlit file. You may add app-specific helper functions where needed for the interface or reasoning trace.

Refer to the provided `StreamlitDeploymentGuide.pdf` guide to deploy your code and include the link for your Streamlit app below.


## Submission

Submit only the following three files:

1. `Sudoku_Assignment.ipynb` — with all required cells run, outputs visible, and all conceptual questions answered.
2. `sudoku_solver.py` — containing your implementation of the knowledge-base and inference functions.
3. `sudoku_app.py` — containing your Streamlit application.

The provided support files are required to run the assignment but are not part of the student submission.

### Deployed Streamlit app URL

[Open the deployed Group 7 Sudoku app](https://daphnezhang627-cell-it5005-group7-sudoku-sudoku-app-lkd5pg.streamlit.app/)

Public functionality checked on 2026-09-25T13:51:15.939Z in a fresh, unauthenticated Edge
browser session. On all five puzzles, both FC and BC produced all 81 reference
values correctly. Solve-time displays, correct/incorrect candidate queries,
proof controls, and clearing results when switching puzzles were checked.
A 390px mobile viewport was also checked. These deployment checks are separate
from the saved independent-Jupyter-kernel experiments above.

**Remaining before submission:** CHOOG SHENG HONG's student ID and confirmed actual contribution statements for all members.
